# 122 — Router y especialistas

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** Matriz (filas = real, columnas = predicho):

```text
           código  datos  docs
código        45      0     5
datos          6     24     0
docs           0      6    14
```

Exactitud = (45+24+14)/100 = **0.83**. Recall: código 45/50 = 0.90,
datos 24/30 = 0.80, docs 14/20 = **0.70** (la clase pequeña, como es típico).

**Ejercicio 2.** Techo = 0.83 × 0.95 ≈ **0.789**. Con router 0.95:
0.95 × 0.95 = **0.9025**. Subir el router 12 puntos sube el techo ~11 puntos: la
mejora del router se transfiere casi íntegra porque multiplica a todo el sistema.

**Ejercicio 3.** Ver celda: reglas ordenadas + `fallback`. Nota que el orden importa
si una frase dispara dos reglas; un router real registra la decisión y su motivo.

**Ejercicio 4.** Con router se ejecuta 1 worker en vez de 3 (~1/3 del coste y de la
latencia agregada), pero se pierden los hallazgos cruzados: en el laboratorio la
decisión final ("mejorar seguridad") depende de comparar los tres scores; un router
solo la produciría si la entrada ya insinuara que seguridad es el problema. Router
optimiza coste cuando las tareas son separables; supervisor-workers optimiza
cobertura cuando la evaluación debe ser integral.


In [ ]:
result = run_lab("multiagent", seed=122)
assert result["kind"] == "multiagent"
assert result["evidence"]
show(result)


In [ ]:
# Ejercicio 1
matriz = [[45, 0, 5], [6, 24, 0], [0, 6, 14]]  # filas real: codigo, datos, docs
clases = ["codigo", "datos", "docs"]
exactitud = sum(matriz[i][i] for i in range(3)) / 100
recalls = {clases[i]: matriz[i][i] / sum(matriz[i]) for i in range(3)}
print("exactitud =", exactitud, "| recalls =", recalls)

# Ejercicio 2
techo = round(exactitud * 0.95, 4)
techo_mejorado = round(0.95 * 0.95, 4)
print("techo =", techo, "→ con router 0.95:", techo_mejorado)

# Ejercicio 3
def route(texto):
    t = texto.lower()
    if any(k in t for k in ("vulnerabilidad", "cve", "exploit")):
        return "security"
    if any(k in t for k in ("readme", "guía")):
        return "documentation"
    if any(k in t for k in ("test", "cobertura")):
        return "quality"
    return "fallback"

for frase in ["Detectamos un CVE crítico", "Falta la guía de instalación",
              "La cobertura de tests bajó", "Hola, ¿qué tal?"]:
    print(f"{frase!r} → {route(frase)}")

# Ejercicio 4
result = run_lab("multiagent", seed=122)
assert result["kind"] == "multiagent"
print([w["agent"] for w in result["result"]["workers"]],
      "→", result["result"]["supervisor"]["decision"])


## Reflexión

1. El laboratorio consolida tres workers fijos; un router elegiría *uno*. ¿Qué información de la entrada necesitarías para decidir entre `quality`, `security` y `documentation`, y qué harías con confianza < umbral?
2. Si la exactitud del router es 0.90 y la del especialista 0.92, el techo es ≈0.828. ¿En cuál de los dos invertirías el siguiente esfuerzo de mejora y con qué evidencia lo decidirías?
3. ¿Por qué "añadir un especialista nuevo" puede *bajar* la exactitud global aunque ese especialista sea excelente?
